# Transit level of service for any area

1. Choose an area: type a place name, draw it on a map, or upload a file.
2. Choose the day, the hours and the transit feeds.
3. Run the study.
4. See the results on an interactive map and download a zip with the streets, the H3 hexagons and the census.

Every input comes from the most local source available: the area's own census and official transit feeds when there are, otherwise global data (WorldPop, Mobility Database, Transitland).

Run the cells in order. In Google Colab: **Runtime → Run all** also works with the defaults (Boston).

## 1. Install

In [ ]:
# Installs the packages (only the missing ones) and gets the study code.
import importlib.util, subprocess, sys
from pathlib import Path

REPO = "https://github.com/CityScope/TransitLOSStudies.git"

if Path("example.py").is_file():                       # running inside TransitLOSStudies/example_metro_area
    HERE = Path.cwd()
else:                                                  # Colab or any other folder: download the study code
    HERE = Path("TransitLOSStudies/example_metro_area")
    if not HERE.is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO])

# requirements.txt, one line per module: only what is not installed yet is installed
requirements = [line.strip() for line in (HERE / "requirements.txt").read_text().splitlines() if line.strip()]
modules = ["geohierarchy", "pycensus", "pyGTFSHandler", "UrbanAccessAnalyzer", "transitlos",
           "rasterio", "folium", "freestiler", "pmtiles", "ipyleaflet", "mapclassify"]
missing = [req for mod, req in zip(modules, requirements) if importlib.util.find_spec(mod) is None]
if missing:
    already = "pandas" in sys.modules
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    if already:
        raise SystemExit("Installed. Restart the session (Runtime → Restart session), then run all cells again.")

sys.path.insert(0, str(HERE.resolve()))
import example

try:                                                   # maps and widgets in Colab
    from google.colab import output, files
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Ready")

## 2. Choose the area

Use **one** of the three options below. The last one you run is used.

### Option A: a place name
`AREA` can be:
- `"municipality"`: the city's own boundary (the official one when the country publishes it)
- `"urban_core"`: the UN urban centre (the dense built-up city)
- `"metro_area"`: the UN metropolitan area (urban centre + commuting zone)

In [ ]:
PLACE = "Boston, Massachusetts, USA"
AREA = "municipality"

aoi = example.area_from_place(PLACE, AREA)
print(aoi["name"].iloc[0], f"{example.area_km2(aoi):,.0f} km²")

### Option B: draw it on a map
Draw a polygon or rectangle with the tools on the left of the map, then run the next cell.

In [ ]:
from ipyleaflet import DrawControl, Map

drawn = []
draw_map = Map(center=(42.36, -71.06), zoom=11, scroll_wheel_zoom=True)
draw = DrawControl(polygon={"shapeOptions": {"color": "#d33"}}, rectangle={"shapeOptions": {"color": "#d33"}},
                   polyline={}, circlemarker={}, marker={}, circle={})
draw.on_draw(lambda control, action, geo_json: drawn.append(geo_json) if action == "created" else None)
draw_map.add(draw)
draw_map

In [ ]:
if drawn:
    aoi = example.area_from_geojson(drawn[-1])
    print(f"Drawn area: {example.area_km2(aoi):,.0f} km²")
else:
    print("Nothing drawn: keeping the area from option A")

### Option C: upload a file
Any file with a boundary: `.gpkg`, `.geojson`, `.kml`, or a zipped shapefile. Set `UPLOAD = True` to pick it from your computer.

In [ ]:
UPLOAD = False        # True: choose a file from your computer (Colab)
AOI_FILE = None       # or the path of a file, e.g. "my_area.gpkg"

if UPLOAD:
    AOI_FILE = next(iter(files.upload()))
if AOI_FILE:
    aoi = example.area_from_file(AOI_FILE)
    print(aoi["name"].iloc[0], f"{example.area_km2(aoi):,.0f} km²")
else:
    print("No file: keeping the area chosen above")

## 3. Check the area

In [ ]:
aoi.explore(style_kwds={"color": "#d33", "fillOpacity": 0.1})

## 4. Choose the day and the hours

In [ ]:
DAY = "weekday"         # "weekday" (a working day), "saturday" or "sunday"
START_TIME = "06:00"
END_TIME = "22:00"

## 5. Choose the transit feeds

- `"download"`: find the area's feeds online: the country's official portal when there is one, plus the feeds it misses from the Mobility Database and Transitland.
  If a source needs an API key, the run asks for it and says where to get one. Press Enter to skip that source.
- A folder with your GTFS feeds, e.g. `"my_feeds/"`.
- A list of feeds: zip files, folders or download links.

In [ ]:
FEEDS = "download"
# FEEDS = "my_feeds/"
# FEEDS = ["bus_gtfs.zip", "https://example.com/rail_gtfs.zip"]

UPLOAD_FEEDS = False    # True: choose GTFS zip files from your computer (Colab)
if UPLOAD_FEEDS:
    FEEDS = list(files.upload())

## 6. Run

This gets the streets, the transit feeds and the census for the area, then computes everything.
A municipality takes from a few minutes to an hour; a whole metro area takes longer.

In [ ]:
zip_path = example.run(aoi, day=DAY, start_time=START_TIME, end_time=END_TIME, feeds=FEEDS)

## 7. See the results on the map

The map shows the finest H3 hexagons or circles and the finest census level, with the stops and routes. The chart button (bottom left) opens the distribution, regression, ANOVA, discretization and metadata tabs.

In [ ]:
example.show_map(zip_path)

## 8. Download the results

The zip holds:
- `streets.parquet`: every street segment with its transit level of service
- `h3_res<N>.parquet`: H3 hexagons at each resolution, with population, level of service and census columns
- `census_<level>.parquet`: the census at its finest level, with the same columns

In [ ]:
if IN_COLAB:
    files.download(str(zip_path))
else:
    print(zip_path)